# AM 11B Lab Activity 3: From Marginal to Total — The Fundamental Theorem of Calculus

**Week 4 — built on: Antiderivatives, Summation, Definite Integration, FTC**

**Authors:** Julie Simons, with Claude (Sonnet 5, Anthropic), via Claude Code — September 2026

*With special thanks to Mariana Bessa and prior AM 11B students, whose earlier notebooks inspired this material, and to Eric Van Dusen and the [Data 88E](https://github.com/data-88e) course for inspiration.*

![Times Square at night, covered in advertising billboards](https://thumb.wikimedia.org/wikipedia/commons/thumb/9/9c/Times_square_at_night.jpg/960px-Times_square_at_night.jpg)

*Photo: Rafi B. from Somewhere in Texas, CC BY 2.0, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Times_square_at_night.jpg)*

<!-- canvas-group-instructions -->
## Group Submission Instructions

**Joining your lab group on Canvas**

1. Agree with your partners on a group number (e.g., "Lab Group 7").
2. In our Canvas course, click **People**, then the **Groups** tab.
3. Find your group and click **Join**. **Every member must do this themselves.** You can't add your partners.
4. To check, click the group name and make sure all your partners are listed. If you joined the wrong group, click **Leave** and join the right one.
5. When you submit, **only one person uploads the notebook**. It counts for everyone in the group.

Please join your group before submitting your first lab. If you submit before joining, only you will get credit.

Please write the name(s) of your group (or yourself) below by double-clicking on this text.

**NAME(S):** *Replace this text with your name(s)*

## Learning Goals

By the end of this lab you should be able to:
1. Compute a small Riemann sum by hand, and see how the rectangle approximation improves as the number of rectangles grows, connecting that limit to the definite integral.
2. Recognize that integrating a marginal (rate) function recovers a total (cumulative) quantity.
3. Find an antiderivative and evaluate a definite integral by hand using the Fundamental Theorem of Calculus, then confirm it both symbolically and numerically in code.
4. Interpret a definite integral as a **shaded area** on a graph of the marginal function.
5. Connect a rate hitting zero to a maximum of the cumulative quantity — a callback to Week 2's optimization work.

### Why economists care
Economists constantly work with *rates* — marginal cost, marginal revenue, marginal profit — because rates are what a decision-maker controls at the margin. But *totals* — total cost, total revenue, total profit — are what actually shows up on a balance sheet. The FTC is the bridge between the two.

## Setup

Run the cell below to load the tools we'll use today.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy import integrate
from ipywidgets import interact, IntSlider
import random

sp.init_printing()
plt.rcParams['figure.figsize'] = (9, 5)
%matplotlib inline

def checkpoint(is_correct, hint=""):
    if is_correct:
        emoji = random.choice(["✅", "🚀", "⭐"])
        msg = random.choice([
            "Great job! You nailed it.",
            "You did it!",
            "Nailed it — nice work.",
            "Yes! That's exactly right.",
            "Correct! You're crushing this.",
        ])
        print(f"{emoji} {msg}")
    else:
        print("❌ Try again!" + (f" {hint}" if hint else ""))

print("Setup complete!")

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

## Warm-Up: Riemann Sums — Building the Definite Integral from Rectangles

Before jumping into the campaign scenario, let's revisit an idea from lecture: a **Riemann sum** approximates the area under a curve by chopping the region into $n$ thin rectangles, computing each rectangle's area, and adding them up:

$$\text{Area} \approx \sum_{i=1}^{n} f(x_i)\, \Delta x$$

As $n \to \infty$ (thinner and thinner rectangles), this sum approaches the *exact* area under the curve — and that limit is exactly how the definite integral $\int_a^b f(x)\,dx$ is defined. The **Fundamental Theorem of Calculus (FTC)**, which you've just seen in lecture, is what lets us skip the tedious process of summing infinitely many rectangles and instead evaluate an antiderivative at the two endpoints.

Let's see this play out with a function you'll recognize from the rest of this lab: $\pi'(t) = 500 - 20t$, a marketing campaign's marginal profit, on the interval $t \in [0, 10]$.

**Before running any code**, compute the $n=2$ left Riemann sum by hand: that's two rectangles, each of width $\Delta t = 5$, with heights given by $\pi'(t)$ at the *left* edge of each rectangle ($t=0$ and $t=5$). Find those two heights, multiply each by the width, and add them up.

</div>

*Replace this text with your hand-computed $n=2$ left Riemann sum.*

Run the cell below to check your hand computation.

In [ ]:
n2_heights = [500 - 20*0, 500 - 20*5]
n2_sum = sum(n2_heights) * 5
print(f"n=2 left Riemann sum: heights {n2_heights}, each rectangle width 5 -> total = ${n2_sum}")

checkpoint(n2_sum == 4500, hint="Heights should be pi'(0)=500 and pi'(5)=400, each times a width of 5.")

Now run the cell below, then **drag the slider to $n=2$** to see the same computation come alive as a picture — then keep dragging it higher and watch the rectangle approximation, and its total area, get closer and closer to the exact area under the curve.

In [ ]:
def mp_rate(t):
    return 500 - 20*t

a, b = 0, 10

def riemann_demo(n=5):
    dt = (b - a) / n
    left_edges = a + dt*np.arange(n)
    heights = mp_rate(left_edges)
    riemann_sum = np.sum(heights * dt)

    t_smooth = np.linspace(a, b, 200)
    plt.figure(figsize=(9, 5))
    plt.plot(t_smooth, mp_rate(t_smooth), 'b-', linewidth=2, label="Marginal profit $\pi'(t)$")
    plt.bar(left_edges, heights, width=dt, align='edge', alpha=0.4, color='orange',
            edgecolor='black', label=f'{n} rectangles (left endpoint)')
    plt.axhline(0, color='gray', linewidth=0.8)
    plt.xlabel('Day (t)')
    plt.ylabel('Marginal profit ($/day)')
    plt.title(f"Left Riemann Sum, n={n} rectangles\nApproximate area = ${riemann_sum:,.2f}")
    plt.legend()
    plt.ylim(0, 550)
    plt.show()

    print(f"n = {n:>4} rectangles  ->  Riemann sum estimate: ${riemann_sum:,.2f}")

interact(riemann_demo, n=IntSlider(min=1, max=200, step=1, value=5, description='n (rectangles)'));

*(If the slider doesn't appear, just edit `value=5` in the cell above to a few different numbers by hand and re-run — the plot and printed estimate will update either way.)*

### Warm-Up Question: Rectangles → Exact Area

As you drag the slider toward larger $n$, what happens to the Riemann sum estimate? What single number does it appear to be converging to? Keep that number in mind — you're about to compute it exactly (and much faster) using the Fundamental Theorem of Calculus.

*Replace this text with your answer.*

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

## The Scenario

A company runs a marketing campaign. Its **marginal profit** — the extra profit earned on day $t$ of the campaign — is:

$$\pi'(t) = 500 - 20t \quad (\text{dollars per day}, \ t \text{ in days})$$

Marginal profit is high on day 1 (the campaign is new and exciting) and declines over time as the audience saturates. Total profit earned from day 0 through day $T$ is:

$$\pi(T) = \int_0^T \pi'(t)\, dt$$

### Step 1: Find total profit over the first 10 days — try it by hand first

**Before running any code**, find the antiderivative $F(t)$ of $\pi'(t) = 500 - 20t$ by hand (basic power rule), then evaluate $F(10) - F(0)$ to get the exact total profit over the first 10 days.

</div>

*Replace this text with your hand-computed antiderivative $F(t)$ and your value of $F(10)-F(0)$.*

We're about to check that by computing the same integral **two different ways** in code, so it's worth pausing on what that means:

- **Symbolic** (what `sympy` does): it works directly with the algebra, finds the actual antiderivative formula, and evaluates it at the endpoints — exactly what you just did by hand. The result is an *exact* answer.
- **Numerical** (what `scipy.integrate.quad` does): it never touches the formula at all. Instead it estimates the area by crunching a huge number of thin slices under the curve — basically a far more precise, automated version of the Riemann sum you just explored with the slider above.

These are two completely independent ways of finding the same number. If they agree with each other — and with your hand computation — that's strong evidence the answer is correct. Run the cell below to see both.

In [ ]:
t = sp.symbols('t')
marginal_profit = 500 - 20*t

total_profit_10 = sp.integrate(marginal_profit, (t, 0, 10))
print(f"Total profit over days 0-10 (symbolic, sympy):     ${total_profit_10}")

# Double-check using a numerical method (scipy) instead of the symbolic one (sympy) above
f = lambda tt: 500 - 20*tt
val, err = integrate.quad(f, 0, 10)
print(f"Total profit over days 0-10 (numerical, scipy):    ${val:.2f}")

checkpoint(
    total_profit_10 == 4000 and abs(val - 4000) < 0.01,
    hint="Total profit over days 0-10 should come out to exactly $4000."
)

Notice: **$4000** is exactly the number the Riemann sum slider above was converging to as $n$ grew large! The FTC gave us that same answer instantly, with no need to sum up hundreds of tiny rectangles by hand — that's the payoff of the theorem.

### Step 2: Visualize it as an area

Run the cell below. The shaded region *is* the definite integral — it's literally the area under the marginal profit curve between $t=0$ and $t=10$.

In [ ]:
t_vals = np.linspace(0, 30, 200)
mp_vals = 500 - 20*t_vals

plt.figure()
plt.plot(t_vals, mp_vals, 'b-', linewidth=2, label="Marginal profit $\pi'(t)$")
plt.axhline(0, color='gray', linewidth=0.8)
t_fill = np.linspace(0, 10, 100)
plt.fill_between(t_fill, 500 - 20*t_fill, alpha=0.3, color='blue', label='Total profit, days 0-10')
plt.xlabel('Day (t)')
plt.ylabel('Marginal profit ($/day)')
plt.title("Marginal Profit Over the Campaign")
plt.legend()
plt.show()

### Question 3.1: Reading the Graph

Notice the marginal profit line crosses zero somewhere past day 20 in the plot above. In words, what does it mean for the *marginal* profit to be negative on a given day? Should the company keep running the campaign once $\pi'(t) < 0$?

*Replace this text with your answer.*

## Finding When Marginal Profit Hits Zero

This connects directly back to Week 2's optimization work: since $\pi'(t) = 0$ marks a critical point of the *total* profit function $\pi(t)$, and marginal profit is decreasing throughout, that critical point is exactly where **total profit is maximized** — the best day to stop the campaign!

Run the cell below to solve for it.

In [ ]:
stop_day = sp.solve(sp.Eq(marginal_profit, 0), t)[0]
total_profit_at_stop = sp.integrate(marginal_profit, (t, 0, stop_day))

print(f"Marginal profit hits zero at t = {stop_day} days")
print(f"Total profit if the campaign runs exactly that long: ${total_profit_at_stop}")

checkpoint(stop_day == 25 and total_profit_at_stop == 6250, hint="Solve 500-20t=0 for t, then integrate up to that t.")

### Seeing It All at Once: Marginal and Total, Side by Side

The relationship between marginal profit and total profit is easiest to see when the two are plotted together. Run the cell below: the **top panel** shows the marginal profit curve $\pi'(t)$ you've been working with all along; the **bottom panel** shows the *running total* $\pi(T)=\int_0^T \pi'(t)\,dt$ as $T$ sweeps from day 0 to day 30.

Watch how the total profit curve climbs fastest while marginal profit is high and positive, keeps climbing (more slowly) as marginal profit shrinks toward zero, and turns over and starts *falling* exactly where marginal profit crosses zero (marked with the dashed green line and dot) — the connection between a rate and its accumulation, made visual.

In [ ]:
F = sp.integrate(marginal_profit, t)
F_numeric = sp.lambdify(t, F, 'numpy')

T_range = np.linspace(0, 30, 200)
running_total = F_numeric(T_range) - F_numeric(0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)

ax1.plot(T_range, 500 - 20*T_range, 'b-', linewidth=2)
ax1.axhline(0, color='gray', linewidth=0.8)
ax1.axvline(float(stop_day), color='green', linestyle=':', label=f'Marginal profit = 0 at t={stop_day}')
ax1.set_ylabel("Marginal profit $\\pi'(t)$ ($/day)")
ax1.set_title("Marginal Profit (top) and Total Profit (bottom)")
ax1.legend()

ax2.plot(T_range, running_total, 'r-', linewidth=2)
ax2.axvline(float(stop_day), color='green', linestyle=':')
ax2.plot([float(stop_day)], [float(total_profit_at_stop)], 'go', markersize=8, label='Maximum total profit')
ax2.set_xlabel('Day (t)')
ax2.set_ylabel('Total profit $\\pi(T)$ ($)')
ax2.legend()

plt.tight_layout()
plt.show()

<div style="background-color:#e7f1fc; border-left:6px solid #4a90d9; border-radius:4px; padding:4px 18px 4px 18px; color:#0b3d78; margin:12px 0;">

## Your Turn

Below, change `T_new` to a **different number of days** (try something less than 25, and then something more than 25, to see both sides) and the code will compute total profit over that window both symbolically and numerically, confirming the two methods agree.

Edit the cell marked `# EDIT HERE`, then run it.

</div>

In [ ]:
T_new = None   # EDIT HERE -- try a few different values, less than and more than 25
assert T_new is not None, "Set T_new to a number before running this cell!"

total_profit_new = sp.integrate(marginal_profit, (t, 0, T_new))
val_new, err_new = integrate.quad(f, 0, T_new)

print(f"Total profit over days 0-{T_new} (sympy):  ${float(total_profit_new):.2f}")
print(f"Total profit over days 0-{T_new} (scipy):  ${val_new:.2f}")

checkpoint(
    abs(float(total_profit_new) - val_new) < 0.01,
    hint="Symbolic and numerical integration should agree -- double check T_new was set correctly."
)

### Question 3.2: Comparing to the Optimal Stopping Point

How does the total profit you just found compare to the maximum total profit at the optimal stopping day (computed above)? Try a value of `T_new` larger than the optimal stopping day — does total profit go up or down? Does that match what you'd expect from a *negative* marginal profit?

*Replace this text with your answer.*

## Reflection

**In 1-2 sentences:** This lab used the Fundamental Theorem of Calculus to go from a rate ($\pi'(t)$) to a total ($\pi(T)$). Give one other economic quantity (besides profit) that is naturally described as a *rate*, and explain what integrating it would give you.

*Replace this text with your answer.*

---

## Submission

1. Make sure you've run all cells in order (**Runtime/Kernel → Restart and run all** is a good final check).
2. Answer the short reflection question(s) above by editing the markdown cells.
3. Go to **File → Download** to save the `.ipynb` file to your computer.
4. Upload the file to Canvas under this Lab Activity.

Only one person per group needs to submit if you worked together — just make sure everyone's name is listed at the top.

**This lab is designed to take about 30 minutes.** If you're stuck for more than a few minutes on any one cell, that's your cue to ask a neighbor, a TA, or post on Ed Discussion — don't burn your whole discussion period on one line of code.
